# Tata Steel — Roller-Table Motor Predictive & Prescriptive Maintenance

**Motor ID:** ROT-MILL-05 | **Process:** TSCR Roller Table | **Sampling Rate:** 1 second

### Pipeline:
1. Full exploratory analysis of all 10 sensor columns
2. Statistical threshold derivation and feature-selection justification
3. ML model training, cross-validation, and benchmarking
4. Health-index scoring and sensor-aware prescriptive maintenance

**Dataset:** `tata_steel_rot_motor_proxy.csv` — 10,000 rows, 1-second intervals, simulated IoT data for a 415 V / 30 kW roller-table motor.

## Part 1 — Exploratory Data Analysis

We load all columns first, understand the data structure, then use correlation and distribution analysis to decide which features matter for modelling.

### 1.1 Data Loading & Overview

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("tata_steel_rot_motor_proxy.csv")

print(f"Shape : {df.shape}")
print(f"Columns: {list(df.columns)}")
print("\n--- Data Types & Nulls ---")
df.info()
print("\n--- Descriptive Statistics (all numeric) ---")
df.describe().round(2)

### 1.2 Correlation Heatmap — All Numeric Sensors

This tells us which sensors move together (shared cause) and which are independent noise. We use this to justify feature selection for the ML model.

In [ ]:
numeric_cols = ["Current_Amp", "Voltage_V", "Motor_RPM", "Vibration_mm_s",
                "Winding_Temp_C", "Bearing_Temp_C", "Coolant_Pressure_Bar", "Ambient_Humidity_Pct"]

plt.figure(figsize=(9, 7))
sns.heatmap(
    df[numeric_cols].corr(),
    annot=True, fmt=".2f", cmap="coolwarm",
    vmin=-1, vmax=1, linewidths=0.5, square=True
)
plt.title("Sensor Correlation Matrix (all 8 numeric columns)", fontsize=13, pad=14)
plt.tight_layout()
plt.show()

**Observations:**
- **Current, RPM, Vibration, and both Temperatures** are correlated — they all respond to the cyclic slab-loading pattern.
- **Voltage, Coolant Pressure, and Humidity** show near-zero correlation with everything else — independent environmental noise.

**Feature selection decision:** `Coolant_Pressure_Bar` and `Ambient_Humidity_Pct` are excluded from the ML feature set. `Voltage` is kept because supply fluctuations can directly affect current draw.

### 1.3 Data Visualization

The motor cycles between idling (~45 A) and loaded (~85 A) as steel slabs pass.

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(12, 4))

axes.plot(df["Current_Amp"].iloc[:500], color="#5B9BD5", linewidth=0.8)
axes.axhline(60, color="gray", linestyle=":", alpha=0.6, label="~60 A midpoint")
axes.set_xlabel("Time Index (seconds)")
axes.set_ylabel("Current (A)")
axes.set_title("Current Over Time — Cyclic Load Switching (first 500 s)")
axes.legend()

plt.tight_layout()
plt.show()

### 1.4 Temperature as a Lagging Indicator

Temperatures change slowly — they smooth towards a target rather than jumping instantly with load. Winding temp responds faster than bearing temp. This makes them good for tracking steady-state health, but not for detecting sudden faults — that role belongs to vibration.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
t = 800

axes[0].plot(df["Current_Amp"].iloc[:t],    color="#5B9BD5", linewidth=0.7, label="Current (A)")
axes[0].plot(df["Winding_Temp_C"].iloc[:t], color="#ED7D31", linewidth=1.2, label="Winding Temp (C)")
axes[0].set_ylabel("Value")
axes[0].set_title("Current vs Winding Temperature — Lagging Response")
axes[0].legend(loc="upper right")

axes[1].plot(df["Current_Amp"].iloc[:t],     color="#5B9BD5", linewidth=0.7, label="Current (A)")
axes[1].plot(df["Bearing_Temp_C"].iloc[:t],  color="#70AD47", linewidth=1.2, label="Bearing Temp (C)")
axes[1].set_xlabel("Time Index (seconds)")
axes[1].set_ylabel("Value")
axes[1].set_title("Current vs Bearing Temperature — Even Slower Response")
axes[1].legend(loc="upper right")

plt.tight_layout()
plt.show()

### 1.5 Rule-Based Anomaly Detection — Vibration Spikes

We can detect simple anomalies by checking if vibration levels exceed a fixed threshold (e.g., 7.0 mm/s).

- **Bearing Temp upper bound:** 85.50 °C
- **Winding Temp upper bound:** 105.61 °C

In [ ]:
# ---- Simple Anomaly Detection (Rule-Based) ----
vibration_threshold = 7.0  # mm/s (abnormally high)

anomalies = df[df["Vibration_mm_s"] > vibration_threshold]

print("Number of vibration anomalies detected:", len(anomalies))
print(anomalies[["Timestamp", "Vibration_mm_s"]].head())

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=False)

# Full timeline
axes[0].plot(df["Vibration_mm_s"], color="#5B9BD5", linewidth=0.6, label="Vibration (mm/s)")
axes[0].scatter(anomalies.index, anomalies["Vibration_mm_s"],
                color="red", s=20, zorder=5, label="Anomaly (spike)")
axes[0].axhline(vibration_threshold, color="orange", linestyle="--",
                label=f"IQR Threshold ({vibration_threshold:.2f})")
axes[0].set_title("Vibration Monitoring — Full Timeline")
axes[0].set_ylabel("Vibration (mm/s)")
axes[0].legend()

# Zoomed
z = 500
zoom_anom = anomalies[anomalies.index < z]
axes[1].plot(df["Vibration_mm_s"].iloc[:z], color="#5B9BD5", linewidth=0.9, label="Vibration (mm/s)")
axes[1].scatter(zoom_anom.index, zoom_anom["Vibration_mm_s"],
                color="red", s=45, zorder=5, label="Anomaly (spike)")
axes[1].axhline(vibration_threshold, color="orange", linestyle="--",
                label=f"IQR Threshold ({vibration_threshold:.2f})")
axes[1].set_title(f"Vibration Monitoring — Zoomed (first {z} s)")
axes[1].set_xlabel("Time Index (seconds)")
axes[1].set_ylabel("Vibration (mm/s)")
axes[1].legend()

plt.tight_layout()
plt.show()

## Part 2: Machine Learning Model
We will train models to predict what the "normal" current should be. If the actual current deviates significantly from the prediction, it may indicate an issue.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

# Feature Selection
features = ["Motor_RPM", "Vibration_mm_s", "Winding_Temp_C", "Bearing_Temp_C", "Voltage_V"]
X = df[features]
y = df["Current_Amp"]

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Training set size:", len(X_train))
print("Test set size:", len(X_test))

### 2.1 Model Training
We'll compare a simple **Decision Tree** against a **Random Forest**. Regression metrics (MAE) will evaluate accuracy.

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# 1. Decision Tree (Baseline)
dt_model = DecisionTreeRegressor(max_depth=5, random_state=42)
dt_model.fit(X_train, y_train)
dt_mae = mean_absolute_error(y_test, dt_model.predict(X_test))
print("Decision Tree MAE (Amps):", round(dt_mae, 2))

# 2. Random Forest (Main Model)
rf_model = RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
rf_mae = mean_absolute_error(y_test, y_pred_rf)
print("Random Forest MAE (Amps):", round(rf_mae, 2))

### 2.2 Health Index & Anomaly Detection
We calculate the **Prediction Error** (Residuals). Large errors mean the motor is behaving unexpectedly (Anomaly). We can map this error to a **Health Index** score (0-1).

In [ ]:
# Calculate Prediction Error
prediction_error = np.abs(y_test - y_pred_rf)

# Dynamic Threshold: Mean + 3 Std Dev
error_threshold = prediction_error.mean() + 3 * prediction_error.std()
ml_anomalies = prediction_error > error_threshold

print(f"Anomaly Threshold (Amps error): {error_threshold:.2f}")
print("Number of ML-detected anomalies:", ml_anomalies.sum())

# Create Health Index (1 = Healthy, 0 = Critical)
normalized_error = prediction_error / prediction_error.max()
health_index = 1 - normalized_error

# Combine into a results DataFrame
health_df = pd.DataFrame({
    "Actual_Current": y_test.values,
    "Predicted_Current": y_pred_rf,
    "Prediction_Error": prediction_error,
    "Health_Index": health_index
})

print("\nHealth Index Summary:")
print(health_df["Health_Index"].describe())

### 2.3 Prescriptive Maintenance
Finally, we categorize the Health Index into status levels and recommend specific maintenance actions.

In [ ]:
def health_status(h):
    if h > 0.8: return "Healthy"
    elif h > 0.6: return "Degrading"
    else: return "Critical"

def prescribe_action(row):
    if row["Health_Status"] == "Healthy": return "No action required"
    elif row["Health_Status"] == "Degrading": return "Schedule inspection and lubrication check"
    else: return "Immediate maintenance: check bearings, alignment, load"

health_df["Health_Status"] = health_df["Health_Index"].apply(health_status)
health_df["Prescriptive_Action"] = health_df.apply(prescribe_action, axis=1)

print("\nHealth Status Counts:")
print(health_df["Health_Status"].value_counts())

print("\nSample Recommendations:")
print(health_df[["Health_Status", "Prescriptive_Action"]].head(10))